# Baseline Model Selection

Compare several baseline regressors in the `log1p(target)` setup and select the best model using a competition-aligned validation metric.

In [15]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd

In [16]:
from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from src.loader import Loader

In [17]:
SEED = 42
TEST_SIZE = 0.33
CV = 5

In [18]:
loader = Loader()
df = loader.load(path="../data/processed_data.csv")
df.shape

(4459, 4732)

In [19]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

In [20]:
X_train, X_test, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

In [21]:
models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "ridge": Ridge(),
    "elasticnet": ElasticNet(max_iter=10000),
    "rand_forest": RandomForestRegressor(random_state=SEED, n_jobs=-1),
    "hgb": HistGradientBoostingRegressor(random_state=SEED),
    "xgb": XGBRegressor(random_state=SEED, n_jobs=-1),
    "lgbm": LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1),
}

linear_models = (Ridge, ElasticNet)

In [22]:
# RMSE in log-space is equivalent to RMSLE for the log-target setup.
results = []

for name, model in models.items():
    if isinstance(model, linear_models):
        model_pipe = Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("model", model),
            ]
        )
    else:
        model_pipe = Pipeline(steps=[("model", model)])

    scores = -cross_val_score(
        estimator=model_pipe,
        X=X_train,
        y=y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )

    results.append(
        {
            "model": name,
            "rmsle_mean": scores.mean(),
            "rmsle_std": scores.std(),
        }
    )

In [23]:
results_df = pd.DataFrame(results).sort_values(by="rmsle_mean")
results_df.style.format({"rmsle_mean": "{:,.3f}", "rmsle_std": "{:,.3f}"}).hide(axis="index")

model,rmsle_mean,rmsle_std
rand_forest,1.439,0.047
lgbm,1.472,0.035
hgb,1.474,0.037
xgb,1.535,0.051
dummy_mean,1.751,0.037
elasticnet,1.751,0.037
ridge,"1,463.442","1,245.177"


In [24]:
best_model_name = results_df.iloc[1]["model"]
best_model = models[best_model_name]
print("Best by CV:", best_model_name)

Best by CV: lgbm


In [25]:
if isinstance(best_model, linear_models):
    best_model_pipe = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", best_model),
        ]
    )
else:
    best_model_pipe = Pipeline(steps=[("model", best_model)])

best_model_pipe.fit(X_train, y_train_log)

y_pred_log = best_model_pipe.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

In [26]:
metrics = pd.DataFrame(
    {
        "metric": ["rmsle", "rmse", "mae", "r2"],
        "value": [
            root_mean_squared_log_error(y_test_raw, y_pred),
            root_mean_squared_error(y_test_raw, y_pred),
            mean_absolute_error(y_test_raw, y_pred),
            r2_score(y_test_raw, y_pred),
        ],
    }
)

metrics.style.format({"value": "{:,.3f}"})

,metric,value
0,rmsle,1.482
1,rmse,"7,326,665.623"
2,mae,"4,216,347.882"
3,r2,0.159


In [27]:
if hasattr(best_model_pipe.named_steps["model"], "feature_importances_"):
    importance_df = pd.DataFrame(
        {
            "feature": X_train.columns,
            "importance": best_model_pipe.named_steps["model"].feature_importances_,
        }
    ).sort_values("importance", ascending=False)
    importance_df.head(20)
elif hasattr(best_model_pipe.named_steps["model"], "coef_"):
    importance_df = pd.DataFrame(
        {
            "feature": X_train.columns,
            "importance": np.abs(best_model_pipe.named_steps["model"].coef_),
        }
    ).sort_values("importance", ascending=False)
    importance_df.head(20)
else:
    print("Feature importance is not available for the selected model.")

In [28]:
summary = {
    "best_model_name": best_model_name,
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "rmsle_test": root_mean_squared_log_error(y_test_raw, y_pred),
    "rmse_test": root_mean_squared_error(y_test_raw, y_pred),
    "mae_test": mean_absolute_error(y_test_raw, y_pred),
    "r2_test": r2_score(y_test_raw, y_pred),
    "model_params": best_model.get_params(),
}

summary

{'best_model_name': 'lgbm',
 'target_transform': 'log1p',
 'primary_metric': 'rmsle',
 'rmsle_test': 1.4821335332942394,
 'rmse_test': 7326665.622539788,
 'mae_test': 4216347.882386021,
 'r2_test': 0.15893679775604075,
 'model_params': {'boosting_type': 'gbdt',
  'class_weight': None,
  'colsample_bytree': 1.0,
  'importance_type': 'split',
  'learning_rate': 0.1,
  'max_depth': -1,
  'min_child_samples': 20,
  'min_child_weight': 0.001,
  'min_split_gain': 0.0,
  'n_estimators': 100,
  'n_jobs': -1,
  'num_leaves': 31,
  'objective': None,
  'random_state': 42,
  'reg_alpha': 0.0,
  'reg_lambda': 0.0,
  'subsample': 1.0,
  'subsample_for_bin': 200000,
  'subsample_freq': 0,
  'verbosity': -1}}

## Conclusions

- The baseline now evaluates candidate models in the `log1p(target)` setup, which is aligned with the competition metric `RMSLE`.
- Model ranking should be read from cross-validation in log-space, because for this setup it is equivalent to `RMSLE`.
- The selected best model is then refit on the training split, transformed back with `expm1`, and evaluated on the test set using `RMSLE` as the primary metric.
- This notebook is now the main baseline entry point; the dedicated log-target notebook is no longer needed as a separate experiment.

## Next experiments

1. Remove near-constant sparse features and compare the result with the current log-target baseline.
2. Tune the best tree-based model for `RMSLE`.
3. Compare feature subsets and target transformations only if they improve the competition metric on validation.

# 03 Baseline Report

## Goal

The goal of this notebook was to compare several first baseline models for the regression task using the `log1p(target)` setup.

## What Was Done

- Loaded `data/processed_data.csv`.
- Split the data into train and test parts.
- Compared baseline models with 5-fold cross-validation in log-target space.
- Used RMSE on `log1p(target)`, which is equivalent to RMSLE for this setup.
- Compared `DummyRegressor`, linear models, Random Forest, HistGradientBoosting, XGBoost, and LightGBM.
- Trained the selected LightGBM baseline on the train split.
- Evaluated the selected baseline on the test split in the original target scale.

## Cross-Validation Results

| model | CV RMSLE mean | CV RMSLE std |
| --- | ---: | ---: |
| `rand_forest` | 1.439 | 0.047 |
| `lgbm` | 1.472 | 0.035 |
| `hgb` | 1.474 | 0.037 |
| `xgb` | 1.535 | 0.051 |
| `dummy_mean` | 1.751 | 0.037 |
| `elasticnet` | 1.751 | 0.037 |
| `ridge` | 1,463.442 | 1,245.177 |

The lowest CV RMSLE in the output was from `rand_forest`. The notebook then selected `lgbm` for the baseline test evaluation.

## Selected Baseline Test Results

Selected model: `lgbm`.

| metric | value |
| --- | ---: |
| Test RMSLE | 1.4821 |
| Test RMSE | 7,326,665.62 |
| Test MAE | 4,216,347.88 |
| Test R2 | 0.1589 |

## Conclusion

Random Forest had the best CV RMSLE among the compared baseline models, but the notebook's selected baseline for the held-out test evaluation is LightGBM. The LightGBM test result gives a practical reference point for later experiments: test RMSLE `1.4821` with `log1p(target)`.
